<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/%E6%A3%80%E6%9F%A5%E8%AE%BE%E8%AE%A1_vs_AF_%E9%A2%84%E6%B5%8B_initial_guess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#@title 检查设计 vs AF 预测
设计文件路径 = "/content/drive/MyDrive/RFdiffusion/candidate_pdb/group4_5.pdb"  #@param {type:"string"}
AF文件路径 = "/content/drive/MyDrive/RFdiffusion/real_seq2.pdb"    #@param {type:"string"}
设计_target链 = "B"  #@param {type:"string"}
设计_binder链 = "A"  #@param {type:"string"}
AF_target链 = "A"    #@param {type:"string"}
AF_binder链 = "B"    #@param {type:"string"}

!pip install -q biopython py3Dmol
from google.colab import drive
drive.mount('/content/drive')

import gzip, io, os
import numpy as np
import py3Dmol
from Bio.PDB import PDBParser, MMCIFParser, PDBIO, Superimposer
from Bio.Data.IUPACData import protein_letters_3to1


def load(path):
    base = path[:-3] if path.endswith(".gz") else path
    parser = MMCIFParser(QUIET=True) if base.endswith(".cif") else PDBParser(QUIET=True)
    if path.endswith(".gz"):
        with gzip.open(path, "rt") as fh:
            return parser.get_structure("s", fh)
    return parser.get_structure("s", path)


def seq_and_ca(chain):
    res = [r for r in chain if r.id[0] == " " and "CA" in r]
    seq = "".join(protein_letters_3to1.get(r.get_resname().capitalize(), "X") for r in res)
    return seq, [r["CA"] for r in res]


def rmsd(a, b):
    xa = np.array([x.coord for x in a]); xb = np.array([x.coord for x in b])
    return float(np.sqrt(((xa - xb) ** 2).sum(axis=1).mean()))


def match_target(d_seq, d_ca, p_seq, p_ca):
    i = p_seq.find(d_seq)
    if i >= 0:
        return d_ca, p_ca[i:i + len(d_ca)]
    i = d_seq.find(p_seq)
    if i >= 0:
        return d_ca[i:i + len(p_ca)], p_ca
    if len(d_seq) == len(p_seq):
        return d_ca, p_ca
    raise ValueError("两边的 target 序列对不上")


def to_pdb_str(struct):
    buf = io.StringIO(); w = PDBIO(); w.set_structure(struct); w.save(buf)
    return buf.getvalue()


D = load(设计文件路径.strip()); P = load(AF文件路径.strip())
dm, pm = D[0], P[0]
def chains_info(model):
    return {ch.id: seq_and_ca(ch) for ch in model if any(r.id[0] == " " for r in ch)}

dinfo, pinfo = chains_info(dm), chains_info(pm)
print("设计文件的链：", {c: len(v[1]) for c, v in dinfo.items()})
print("AF 文件的链：", {c: len(v[1]) for c, v in pinfo.items()})

ps, pt_ca = pinfo[AF_target链.strip()]
_, pb_ca = pinfo[AF_binder链.strip()]

# 设计的 binder = 长度和 AF binder 相同的那条链；target = 剩下的链里最长的
cands = [c for c, v in dinfo.items() if len(v[1]) == len(pb_ca)]
if len(cands) != 1:
    raise ValueError(f"设计文件里找不到唯一一条 {len(pb_ca)} 个残基的链，请把上面打印的链信息贴给我")
设计_binder链 = cands[0]
设计_target链 = max((c for c in dinfo if c != 设计_binder链), key=lambda c: len(dinfo[c][1]))
print(f"自动识别：设计 binder = 链 {设计_binder链}，设计 target = 链 {设计_target链}")
ds, dt_ca = dinfo[设计_target链]
_, db_ca = dinfo[设计_binder链]
dt_ca, pt_ca = match_target(ds, dt_ca, ps, pt_ca)

sup = Superimposer(); sup.set_atoms(pt_ca, dt_ca)      # 只用 target 叠合
target_fit = float(sup.rms)
tmp = [a.copy() for a in db_ca]
for a in tmp:
    a.transform(*sup.rotran)
placement = rmsd(tmp, pb_ca)

sup2 = Superimposer(); sup2.set_atoms(pb_ca, [a.copy() for a in db_ca])
fold = float(sup2.rms)

print(f"target_fit        = {target_fit:.2f} Å   (应 < 1，否则对应关系错了，下面两个数作废)")
print(f"binder_placement  = {placement:.2f} Å   (大 = AF 把 binder 放到了别处)")
print(f"binder_fold       = {fold:.2f} Å   (大 = AF 没折成设计的形状)")

D.transform(*sup.rotran)                                # 整个设计结构挪到 AF 坐标系
out_path = os.path.splitext(设计文件路径.strip().replace(".gz", ""))[0] + "_aligned.pdb"
w = PDBIO(); w.set_structure(D); w.save(out_path)
print("已保存叠合后的设计结构：", out_path)

v = py3Dmol.view(width=750, height=520)
v.addModel(to_pdb_str(P), "pdb"); v.addModel(to_pdb_str(D), "pdb")
v.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})
v.setStyle({"model": 0, "chain": AF_binder链.strip()}, {"cartoon": {"color": "green"}})
v.setStyle({"model": 1}, {})
v.setStyle({"model": 1, "chain": 设计_binder链.strip()}, {"cartoon": {"color": "orange"}})
v.zoomTo()
print("灰色 = target，绿色 = AF 预测的 binder，橙色 = 设计的 binder")
v.show()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
设计文件的链： {'A': 63, 'B': 137}
AF 文件的链： {'A': 137, 'B': 63}
自动识别：设计 binder = 链 A，设计 target = 链 B
target_fit        = 0.41 Å   (应 < 1，否则对应关系错了，下面两个数作废)
binder_placement  = 48.04 Å   (大 = AF 把 binder 放到了别处)
binder_fold       = 1.85 Å   (大 = AF 没折成设计的形状)
已保存叠合后的设计结构： /content/drive/MyDrive/RFdiffusion/candidate_pdb/group4_5_aligned.pdb
灰色 = target，绿色 = AF 预测的 binder，橙色 = 设计的 binder


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
#@title 查看文件里有哪些链
路径 = "/content/drive/MyDrive/RFdiffusion/candidate_pdb"  #@param {type:"string"}

!pip install -q biopython
from google.colab import drive
drive.mount('/content/drive')

import os, glob, gzip
from Bio.PDB import PDBParser, MMCIFParser
from Bio.Data.IUPACData import protein_letters_3to1

def load(path):
    base = path[:-3] if path.endswith(".gz") else path
    parser = MMCIFParser(QUIET=True) if base.endswith(".cif") else PDBParser(QUIET=True)
    if path.endswith(".gz"):
        with gzip.open(path, "rt") as fh:
            return parser.get_structure("s", fh)
    return parser.get_structure("s", path)

路径 = 路径.strip()
if os.path.isdir(路径):
    files = sorted(f for ext in ("cif", "pdb", "cif.gz", "pdb.gz") for f in glob.glob(os.path.join(路径, "*." + ext)))
else:
    files = [路径]

for f in files:
    print("==", os.path.basename(f))
    try:
        for ch in load(f)[0]:
            res = [r for r in ch if r.id[0] == " "]
            if not res:
                continue
            seq = "".join(protein_letters_3to1.get(r.get_resname().capitalize(), "X") for r in res)
            print(f"   链 {ch.id}: {len(res)} 个残基, 每个残基约 {len(list(res[0]))} 个原子, 序列 {seq[:25]}{'...' if len(seq) > 25 else ''}")
    except Exception as e:
        print("   读取失败：", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
== group4_5.pdb
   链 A: 63 个残基, 每个残基约 4 个原子, 序列 GGGGGGGGGGGGGGGGGGGGGGGGG...
   链 B: 137 个残基, 每个残基约 4 个原子, 序列 LHILPVAFRGDSFTHTPPLDPQELD...
== run1_model_1.cif.gz
   链 A: 172 个残基, 每个残基约 11 个原子, 序列 RKVCNGIGIGEFKDSLSINATNIKH...
   链 B: 70 个残基, 每个残基约 6 个原子, 序列 SSISLDSWAAAVDTHLATGTVAAAI...
== run1_model_5.cif.gz
   链 A: 172 个残基, 每个残基约 11 个原子, 序列 RKVCNGIGIGEFKDSLSINATNIKH...
   链 B: 70 个残基, 每个残基约 7 个原子, 序列 PGANFFNQDEVTALVANASAKAKTA...
== run1_model_7.cif.gz
   链 A: 172 个残基, 每个残基约 11 个原子, 序列 RKVCNGIGIGEFKDSLSINATNIKH...
   链 B: 70 个残基, 每个残基约 5 个原子, 序列 AQIAALVKNIETSSHPIGPAISNDE...
== run2_model_0.cif.gz
   链 A: 172 个残基, 每个残基约 11 个原子, 序列 RKVCNGIGIGEFKDSLSINATNIKH...
   链 B: 61 个残基, 每个残基约 5 个原子, 序列 AEKLLEYIGDLIDPNFASKVSIDKI...
== run2_model_2.cif.gz
   链 A: 172 个残基, 每个残基约 11 个原子, 序列 RKVCNGIGIGEFKDSLSINATNIKH...
   链 B: 61 个残基, 每个残基约 8 个原子, 序列 NAAQTAGSLPVATLSPAETAYANIT.

In [8]:
!pip install -q git+https://github.com/sokrypton/ColabDesign.git
from colabdesign import mk_afdesign_model
m = mk_afdesign_model(protocol="binder")
help(m.prep_inputs)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
Help on method _prep_binder in module colabdesign.af.prep:

_prep_binder(
    pdb_filename,
    target_chain='A',
    binder_len=50,
    rm_target=False,
    rm_target_seq=False,
    rm_target_sc=False,
    binder_chain=None,
    rm_binder=True,
    rm_binder_seq=True,
    rm_binder_sc=True,
    rm_template_ic=False,
    hotspot=None,
    ignore_missing=True,
    **kwargs
) method of colabdesign.af.model.mk_af_model instance
    prep inputs for binder design
    ---------------------------------------------------
    -binder_len = length of binder to hallucinate (option ignored if binder_c

In [6]:
#@title Initial guess 验证（真序列 vs 打乱序列）
设计文件路径 = "/content/drive/MyDrive/RFdiffusion/candidate_pdb/group4_5.pdb"  #@param {type:"string"}
设计_target链 = "B"  #@param {type:"string"}
设计_binder链 = "A"  #@param {type:"string"}
binder序列 = "DEEEAALAAELEPELAALREEHAALPEAEREALIAAYRAARRAERAAREAAARAAEELAAAAA"  #@param {type:"string"}

import os, random, gzip
!pip install -q biopython git+https://github.com/sokrypton/ColabDesign.git
from google.colab import drive
drive.mount('/content/drive')

# AF 参数（已经有 params 文件夹就会跳过）
if not os.path.exists("params/params_model_1_multimer_v3.npz"):
    !mkdir -p params
    !curl -fsSL https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar | tar x -C params

from Bio.PDB import PDBParser, MMCIFParser, PDBIO
from colabdesign.af import mk_af_model

# 1) 统一转成 PDB（避免 cif 兼容问题）
p = 设计文件路径.strip()
base = p[:-3] if p.endswith(".gz") else p
parser = MMCIFParser(QUIET=True) if base.endswith(".cif") else PDBParser(QUIET=True)
if p.endswith(".gz"):
    with gzip.open(p, "rt") as fh:
        s = parser.get_structure("s", fh)
else:
    s = parser.get_structure("s", p)
tmp_pdb = "/content/design_for_af.pdb"
io = PDBIO(); io.set_structure(s); io.save(tmp_pdb)

# 2) 建模型：multimer + initial guess（保持和之前复合物验证一致，multimer 与否请自己确认）
model = mk_af_model(protocol="binder", use_multimer=True,
                    use_initial_guess=True, use_initial_atom_pos=True,
                    num_recycles=3, data_dir=".")
model.prep_inputs(pdb_filename=tmp_pdb,
                  target_chain=设计_target链.strip(),
                  binder_chain=设计_binder链.strip(),
                  rm_binder=False, rm_binder_seq=True, rm_binder_sc=True)

def run(seq, name):
    model.predict(seq=seq, models=[0, 1, 2, 3, 4], num_models=5, num_recycles=3, verbose=False)
    log = model.aux["log"]
    nums = {k: round(float(v), 3) for k, v in log.items() if isinstance(v, (int, float))}
    print(name, nums)
    model.save_pdb(f"/content/{name}.pdb")
    return nums

seq = binder序列.strip()
shuf = list(seq); random.seed(0); random.shuffle(shuf); shuf = "".join(shuf)
print("binder 长度：", len(seq))
r_real = run(seq, "real_seq")
r_shuf = run(shuf, "shuffled_seq")
print("\n对比 i_ptm：真序列", r_real.get("i_ptm"), " 打乱序列", r_shuf.get("i_ptm"))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
binder 长度： 63
real_seq {'con': 3.213, 'dgram_cce': 228.7, 'exp_res': 0.031, 'fape': 154.62, 'i_con': 4.411, 'i_pae': 0.74, 'pae': 0.56, 'plddt': 0.59, 'rmsd': 39.895, 'seq_ent': 2.713, 'loss': 228.7, 'i_ptm': 0.119, 'ptm': 0.638, 'hard': 1.0, 'soft': 0.0, 'temp': 1.0, 'seqid': 0.0, 'recycles': 3.0}
shuffled_seq {'con': 4.462, 'dgram_cce': 313.949, 'exp_res': 0.238, 'fape': 182.638, 'i_con': 4.405, 'i_pae': 0.778, 'pae': 0.685, 'plddt': 0.402, 'rmsd': 45.742, 'seq_ent': 2.713, 'loss': 313.949, 'i_ptm': 0.115, 'ptm': 0.639, 'hard': 1.0, 'soft': 0.0, 'temp': 1.0, 'seqid': 0.0, 'recycles': 3.0}

对比 i_ptm：真序列 0.119  打乱序列 0.115
